# 02 — Tone classifier

Trains and inspects the softmax logistic-regression tone classifier (Section 6.8) on the
800-row labeled set `data/tone_training.csv`: accuracy, confusion matrix, and example
predictions for the 8 tone labels. Weights are cached to `tone_training_model.npz`.

In [ ]:
import sys, pathlib, numpy as np
sys.path.insert(0, str(pathlib.Path('..', 'backend').resolve()))

from app.ai.embedder import get_embedder
from app.ai.tone import load_or_train, confusion_matrix
from app.ai.text import normalise

emb = get_embedder()
csv_path = str(pathlib.Path('..', 'data', 'tone_training.csv'))
clf = load_or_train(csv_path, emb)       # loads the .npz cache, trains if missing
print("labels:", clf.labels)

examples = [
    "Start your morning right - our baristas can't wait to meet you!",
    "Last call: November's seasonal menu ends Sunday.",
    "Three brewing ratios every coffee lover should know.",
]
E = np.asarray(emb.embed([normalise(t) for t in examples]))  # trainer normalises first
for text, lab, p in zip(examples, clf.predict(E), clf.proba(E).max(axis=1)):
    print(f"{lab:>10}  {p:.2f}  {text[:60]}")

In [ ]:
# Confusion matrix on the training set (Section 17 sanity check)
import csv
rows = list(csv.DictReader(open(csv_path, encoding='utf-8')))
y_true = [r['tone'] for r in rows]
y_pred = clf.predict(np.asarray(emb.embed([normalise(r['text']) for r in rows])))
cm = confusion_matrix(y_true, y_pred, labels=clf.labels)
M = np.asarray(cm['matrix'])
print('accuracy:', round(float(np.trace(M) / M.sum()), 3))
print(cm)